In [5]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50, MobileNetV2, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [9]:
from tensorflow.keras.applications.efficientnet import preprocess_input

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input, ##changed 
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2] ##Added , removed shear range
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [10]:
train_generator = train_datagen.flow_from_directory(
    "split_dataset/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    "split_dataset/val",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    "split_dataset/test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

num_classes = train_generator.num_classes

Found 3323 images belonging to 9 classes.
Found 712 images belonging to 9 classes.
Found 717 images belonging to 9 classes.


In [18]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import numpy as np

def evaluate_model_full(model, model_name):
    
    # Predictions
    y_pred_probs = model.predict(test_generator)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    # True labels
    y_true = test_generator.classes
    class_labels = list(test_generator.class_indices.keys())

    # Metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted')
    rec = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')

    print(f"\n📊 {model_name} RESULTS")
    print("="*40)
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    print("\n📄 Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_labels))

    return acc, prec, rec, f1

**EfficientNetB0**

In [14]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model

input_tensor = layers.Input(shape=(224,224,3))

base = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=input_tensor)

for layer in base.layers:
    layer.trainable = False

x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dense(256, activation='relu')(x)
output = layers.Dense(num_classes, activation='softmax')(x)

baseline_model = Model(inputs=input_tensor, outputs=output)

baseline_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

baseline_model.fit(train_generator, validation_data=val_generator, epochs=10)

evaluate_model(baseline_model, test_generator, "EfficientNetB0")

Epoch 1/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 229s 2s/step - accuracy: 0.6807 - loss: 0.9061 - val_accuracy: 0.7921 - val_loss: 0.5867
Epoch 2/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 184s 2s/step - accuracy: 0.8270 - loss: 0.5015 - val_accuracy: 0.8301 - val_loss: 0.4918
Epoch 3/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - accuracy: 0.8697 - loss: 0.3706 - val_accuracy: 0.8272 - val_loss: 0.4836
Epoch 4/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 182s 2s/step - accuracy: 0.8893 - loss: 0.3188 - val_accuracy: 0.8287 - val_loss: 0.4586
Epoch 5/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 186s 2s/step - accuracy: 0.9094 - loss: 0.2614 - val_accuracy: 0.8427 - val_loss: 0.4799
Epoch 6/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 185s 2s/step - accuracy: 0.9350 - loss: 0.2010 - val_accuracy: 0.8469 - val_loss: 0.4007
Epoch 7/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 185s 2s/step - accuracy: 0.9425 - loss: 0.1728 - val_accuracy: 0.8638 - val_loss: 0.3863
Epoch 8/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 175s 2s/step - accuracy: 0.9488 - loss: 0.1584 - val_accu

In [15]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

# Input
input_tensor = layers.Input(shape=(224,224,3))

# Base model
base = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=input_tensor)

# Freeze layers
for layer in base.layers:
    layer.trainable = False

# Head
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(num_classes, activation='softmax')(x)

# Model
baseline_model = Model(inputs=input_tensor, outputs=output)

# Compile
baseline_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', Precision(name='precision'), Recall(name='recall')]
)

# Callback
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train
history_baseline = baseline_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 198s 2s/step - accuracy: 0.2916 - loss: 2.5970 - precision: 0.3595 - recall: 0.2010 - val_accuracy: 0.5478 - val_loss: 1.5866 - val_precision: 0.8953 - val_recall: 0.1081
Epoch 2/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 172s 2s/step - accuracy: 0.5053 - loss: 1.8194 - precision: 0.5997 - recall: 0.4063 - val_accuracy: 0.6742 - val_loss: 1.2905 - val_precision: 0.8560 - val_recall: 0.4340
Epoch 3/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 170s 2s/step - accuracy: 0.5835 - loss: 1.5918 - precision: 0.6766 - recall: 0.4905 - val_accuracy: 0.7177 - val_loss: 1.1747 - val_precision: 0.8390 - val_recall: 0.5857
Epoch 4/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 178s 2s/step - accuracy: 0.6440 - loss: 1.4457 - precision: 0.7338 - recall: 0.5624 - val_accuracy: 0.7388 - val_loss: 1.1524 - val_precision: 0.8373 - val_recall: 0.6503
Epoch 5/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 167s 2s/step - accuracy: 0.6681 - loss: 1.3666 - precision: 0.7501 - recall: 0.5916 - val_accuracy: 0.7612 - val_los

In [19]:
evaluate_model_full(baseline_model, "EfficientNetB0")

23/23 ━━━━━━━━━━━━━━━━━━━━ 28s 1s/step

📊 EfficientNetB0 RESULTS
Accuracy : 0.7852
Precision: 0.7840
Recall   : 0.7852
F1-score : 0.7808

📄 Classification Report:
                     precision    recall  f1-score   support

          Cardboard       0.76      0.93      0.83        70
      Food Organics       0.74      0.81      0.77        62
              Glass       0.78      0.86      0.82        63
              Metal       0.81      0.85      0.83       119
Miscellaneous Trash       0.65      0.48      0.55        75
              Paper       0.88      0.76      0.81        75
            Plastic       0.81      0.73      0.77       139
      Textile Trash       0.73      0.79      0.76        48
         Vegetation       0.84      0.92      0.88        66

           accuracy                           0.79       717
          macro avg       0.78      0.79      0.78       717
       weighted avg       0.78      0.79      0.78       717



(0.7852161785216178, 0.783998224669176, 0.7852161785216178, 0.7808464637944094)

**MULTI-MODEL (NO CBAM)**

In [21]:
from tensorflow.keras.applications import ResNet50, MobileNetV2, EfficientNetB0

# Input
input_tensor = layers.Input(shape=(224,224,3))

# Base models
resnet = ResNet50(weights='imagenet', include_top=False, input_tensor=input_tensor)
mobilenet = MobileNetV2(weights='imagenet', include_top=False, input_tensor=input_tensor)
efficientnet = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=input_tensor)

# Freeze all layers
for model in [resnet, mobilenet, efficientnet]:
    for layer in model.layers:
        layer.trainable = False

# Feature extraction
r = layers.GlobalAveragePooling2D()(resnet.output)
m = layers.GlobalAveragePooling2D()(mobilenet.output)
e = layers.GlobalAveragePooling2D()(efficientnet.output)

# Fusion
fused = layers.Concatenate()([r, m, e])

# Head
x = layers.Dense(256, activation='relu')(fused)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(num_classes, activation='softmax')(x)

# Model
multi_no_cbam_model = Model(inputs=input_tensor, outputs=output)

# Compile
multi_no_cbam_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', Precision(name='precision'), Recall(name='recall')]
)

# Train
history_multi = multi_no_cbam_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[early_stop]
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_10536\4128713820.py:8: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  mobilenet = MobileNetV2(weights='imagenet', include_top=False, input_tensor=input_tensor)


Epoch 1/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 432s 4s/step - accuracy: 0.4348 - loss: 2.0097 - precision: 0.5518 - recall: 0.3352 - val_accuracy: 0.5955 - val_loss: 1.4156 - val_precision: 0.7128 - val_recall: 0.4846
Epoch 2/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 383s 4s/step - accuracy: 0.6581 - loss: 1.3880 - precision: 0.7445 - recall: 0.5489 - val_accuracy: 0.7205 - val_loss: 1.1602 - val_precision: 0.8225 - val_recall: 0.6053
Epoch 3/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 411s 4s/step - accuracy: 0.7066 - loss: 1.2383 - precision: 0.7936 - recall: 0.6214 - val_accuracy: 0.7767 - val_loss: 1.1010 - val_precision: 0.8613 - val_recall: 0.6713
Epoch 4/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 432s 4s/step - accuracy: 0.7478 - loss: 1.1330 - precision: 0.8326 - recall: 0.6645 - val_accuracy: 0.7893 - val_loss: 1.0567 - val_precision: 0.8567 - val_recall: 0.7135
Epoch 5/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 493s 5s/step - accuracy: 0.7550 - loss: 1.1105 - precision: 0.8344 - recall: 0.6792 - val_accuracy: 0.7963 - val_los

In [22]:
y_pred_multi = np.argmax(multi_no_cbam_model.predict(test_generator), axis=1)

print(classification_report(y_true, y_pred_multi, target_names=list(test_generator.class_indices.keys())))

23/23 ━━━━━━━━━━━━━━━━━━━━ 98s 4s/step
                     precision    recall  f1-score   support

          Cardboard       0.69      0.76      0.72        70
      Food Organics       0.67      0.79      0.73        62
              Glass       1.00      0.29      0.44        63
              Metal       0.90      0.51      0.65       119
Miscellaneous Trash       0.83      0.07      0.12        75
              Paper       1.00      0.37      0.54        75
            Plastic       0.50      0.92      0.65       139
      Textile Trash       0.38      0.96      0.54        48
         Vegetation       0.84      0.86      0.85        66

           accuracy                           0.62       717
          macro avg       0.76      0.61      0.58       717
       weighted avg       0.75      0.62      0.59       717



In [23]:
evaluate_model_full(multi_no_cbam_model, "Multi-No-CBAM")

23/23 ━━━━━━━━━━━━━━━━━━━━ 84s 4s/step

📊 Multi-No-CBAM RESULTS
Accuracy : 0.6206
Precision: 0.7527
Recall   : 0.6206
F1-score : 0.5902

📄 Classification Report:
                     precision    recall  f1-score   support

          Cardboard       0.69      0.76      0.72        70
      Food Organics       0.67      0.79      0.73        62
              Glass       1.00      0.29      0.44        63
              Metal       0.90      0.51      0.65       119
Miscellaneous Trash       0.83      0.07      0.12        75
              Paper       1.00      0.37      0.54        75
            Plastic       0.50      0.92      0.65       139
      Textile Trash       0.38      0.96      0.54        48
         Vegetation       0.84      0.86      0.85        66

           accuracy                           0.62       717
          macro avg       0.76      0.61      0.58       717
       weighted avg       0.75      0.62      0.59       717



(0.6206415620641562,
 0.7527192727892899,
 0.6206415620641562,
 0.5901537377775613)